# Demo 4: Datan puhdistus ja esikäsittely

Tässä demossa käsitellään valmiiksi annettua myyntiaineistoa. Tavoitteena on tunnistaa datan laatuongelmia, korjata niitä ja tallentaa puhdistettu aineisto analysointia varten.


## Tehtävä 1: Lataa aineisto

Lataa annettu CSV-tiedosto Pandas-dataframeen. Tulosta aineiston ensimmäiset rivit sekä selvitä rivien ja sarakkeiden määrä.


In [ ]:
import pandas as pd

# Vaihda tiedostonimi tarvittaessa oman tiedostosi nimeksi
df = pd.read_csv("~/shared/myyntidata.csv")  # ~/shared viittaa Noppe-ympäristön shared-kansioon

display(df.head())
print("Rivejä ja sarakkeita:", df.shape)


## Tehtävä 2: Tutki aineiston rakennetta

Tutki aineiston sarakkeita, tietotyyppejä ja sisältöä. Selvitä, havaitsetko aineistossa ongelmia, jotka pitäisi korjata ennen analysointia.


In [ ]:
print(df.columns)
print()
df.info()

display(df.describe(include="all"))


## Tehtävä 3: Käsittele puuttuvat arvot

Selvitä, missä sarakkeissa on puuttuvia arvoja ja kuinka paljon niitä on. Korvaa tai poista puuttuvia arvoja tarkoituksenmukaisella tavalla.


In [ ]:
# Puuttuvien arvojen määrä sarakkeittain
print(df.isna().sum())

# Esimerkki: täytetään puuttuvat asiakastiedot
if "Asiakas" in df.columns:
    df["Asiakas"] = df["Asiakas"].fillna("Tuntematon")

# Esimerkki: jos hinta tai määrä puuttuu, poistetaan kyseinen havainto
pakolliset = [sarake for sarake in ["Määrä", "Yksikköhinta"] if sarake in df.columns]
if pakolliset:
    df = df.dropna(subset=pakolliset)

print(df.isna().sum())


## Tehtävä 4: Käsittele duplikaatit

Selvitä, sisältääkö aineisto samoja kirjauksia useampaan kertaan. Tutki duplikaatit ja poista tarpeettomat kaksoiskappaleet. Alla oleva koodi tutkii ovatko jotkin rivit täsmälleen samanlaisia ja poistaa kaksoiskappaleet.


In [ ]:
print("Duplikaatteja:", df.duplicated().sum())

display(df[df.duplicated(keep=False)].head(20))

df = df.drop_duplicates()
print("Duplikaatteja poiston jälkeen:", df.duplicated().sum())


## Tehtävä 5: Korjaa tietotyypit

Tarkista sarakkeiden tietotyypit. Etsi sarakkeita, joiden tietotyyppi ei vastaa niiden sisältämää tietoa. Muuta tietotyypit sopiviksi.


In [ ]:
print(df.dtypes)

In [ ]:
# Päivämäärä päivämäärämuotoon
df["Päivämäärä"] = pd.to_datetime(
    df["Päivämäärä"],
    format="%d.%m.%Y",
    errors="coerce"
)

In [ ]:
#Yksikköhinta ei ole numeerinen. Tutkitaan mistä tämä voisi johtua kokeilemalla muuttaa arvot numeerisiksi. 

hinnat_numeroina = pd.to_numeric(
    df["Yksikköhinta"],
    errors="coerce"
)

#Tulostetaan sekä NaN että tekstiarvot, joita ei voitu muuttaa numeroiksi. Nyt huomataan, että osassa lukuja on desimaalipilkku, kun 
#Pythonissa pitää käyttää pistettä. 

df[hinnat_numeroina.isna()]



In [ ]:
# Korjataan yksikköhinnan desimaalierotin ja muutetaan se numeeriseksi. 
df["Yksikköhinta"] = (
    df["Yksikköhinta"]
    .astype("string")
    .str.replace(",", ".", regex=False)
)

df["Yksikköhinta"] = pd.to_numeric(
    df["Yksikköhinta"],
    errors="coerce"
)

In [ ]:
print(df.dtypes)

## Tehtävä 6: Siisti tekstimuotoista dataa

Tutki tekstimuotoisten sarakkeiden arvoja. Korjaa esimerkiksi ylimääräiset välilyönnit, kirjainkoko ja saman asian erilaiset kirjoitusasut.


In [ ]:
# Tarkastellaan ensin yksilöllisiä arvoja
for sarake in ["Alue", "Tuoteryhmä"]:
    if sarake in df.columns:
        print(sarake, df[sarake].dropna().unique())

# Poistetaan ylimääräiset välilyönnit ja yhdenmukaistetaan kirjoitusasut
for sarake in ["Alue", "Tuote", "Tuoteryhmä", "Asiakas"]:
    if sarake in df.columns:
        df[sarake] = df[sarake].astype("string").str.strip()

if "Alue" in df.columns:
    df["Alue"] = df["Alue"].str.title()

if "Tuoteryhmä" in df.columns:
    df["Tuoteryhmä"] = df["Tuoteryhmä"].str.title()

for sarake in ["Alue", "Tuoteryhmä"]:
    if sarake in df.columns:
        print(sarake, df[sarake].dropna().unique())


## Tehtävä 7: Tarkista poikkeavat arvot

Tutki numeeristen muuttujien arvoja. Etsi selvästi virheellisiä tai epärealistisia arvoja ja käsittele ne tarkoituksenmukaisesti.


In [ ]:
sarakkeet = [sarake for sarake in ["Määrä", "Yksikköhinta"] if sarake in df.columns]
if sarakkeet:
    display(df[sarakkeet].describe())

# Esimerkkisääntö: määrän ja yksikköhinnan täytyy olla positiivisia
if "Määrä" in df.columns:
    display(df[df["Määrä"] <= 0])
    df = df[df["Määrä"] > 0]

if "Yksikköhinta" in df.columns:
    display(df[df["Yksikköhinta"] <= 0])
    df = df[df["Yksikköhinta"] > 0]


## Tehtävä 8: Tarkista puhdistettu aineisto

Tutki aineisto uudelleen ja varmista, että tekemäsi korjaukset ovat onnistuneet. Vertaa havaintojen määrää ja aineiston rakennetta alkuperäiseen tilanteeseen.


In [ ]:
df.info()
print()
print("Rivejä ja sarakkeita puhdistuksen jälkeen:", df.shape)
print()
print("Puuttuvat arvot:")
print(df.isna().sum())
print()
print("Duplikaatteja:", df.duplicated().sum())


## Tehtävä 9: Tallenna puhdistettu aineisto

Tallenna puhdistettu dataframe uutena CSV-tiedostona siten, että alkuperäinen tiedosto säilyy muuttumattomana.


In [ ]:
df.to_csv("~/myyntidata_puhdistettu.csv", index=False, encoding="utf-8-sig") # ~/ viittaa Noppe-ympäristön kotikansioosi


## Tehtävä 10: Visualisoi myynti

Laske ensin rivikohtainen myynti. Tee puhdistetusta aineistosta:

- pinottu pylväskaavio, joka esittää myyntiä alueittain ja tuoteryhmittäin
- ympyräkaavio, joka esittää tuoteryhmien osuudet kokonaismyynnistä.

Lisää kaavioihin tarkoituksenmukaiset otsikot yms.


In [ ]:
import matplotlib.pyplot as plt

# Rivikohtainen myynti
df["Myynti"] = df["Määrä"] * df["Yksikköhinta"]

# Pinottu pylväskaavio
myynti_alueittain = df.pivot_table(
    index="Alue",
    columns="Tuoteryhmä",
    values="Myynti",
    aggfunc="sum"
)

myynti_alueittain.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

plt.title("Myynti alueittain ja tuoteryhmittäin")
plt.xlabel("Alue")
plt.ylabel("Myynti")
plt.legend(title="Tuoteryhmä")
plt.tight_layout()
plt.show()


In [ ]:
# Ympyräkaavio tuoteryhmien osuuksista
myynti_tuoteryhmittain = df.groupby("Tuoteryhmä")["Myynti"].sum()

myynti_tuoteryhmittain.plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(7, 7)
)

plt.title("Tuoteryhmien osuudet kokonaismyynnistä")
plt.ylabel("")
plt.tight_layout()
plt.show()
